In [0]:
from pyspark.sql import functions as F

# 1. Configuração do Widget de Ambiente
dbutils.widgets.dropdown("env", "homolog", ["dev", "homolog", "prod"], "Ambiente")
ENV = dbutils.widgets.get("env")

# 2. Resolução dos schemas do ambiente
SCHEMA_RAW = f"workspace.{ENV}_raw"
SCHEMA_SILVER = f"workspace.{ENV}_silver"
SCHEMA_GOLD = f"workspace.{ENV}_gold"

print(f"=== INICIANDO INGESTÃO RAW PARA O AMBIENTE: [{ENV.upper()}] ===")
print(f"Target Schema Raw: {SCHEMA_RAW}")

# 3. Garantir a existência do schema no Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_RAW}")

# 4. Caminho base do Volume compartilhado
BASE_VOLUME = "/Volumes/workspace/default/cnpj_ingestion/bronze"

# Definição do limite de linhas por ambiente para otimização de custos e performance
LIMITES_AMOSTRA = {
    "dev": 1000,
    "homolog": 10000,
    "prod": None,  # Base completa
}
limite_linhas = LIMITES_AMOSTRA.get(ENV)

# =============================================================================
# 5. INGESTÃO DAS ENTIDADES PRINCIPAIS (Empresas, Estabelecimentos, Sócios)
# =============================================================================
tabelas_principais = ["empresas", "estabelecimentos", "socios"]

for entidade in tabelas_principais:
    caminho = f"{BASE_VOLUME}/{entidade}"
    print(f"Processando entidade: {entidade} a partir de {caminho}...")

    df_raw = (
        spark.read.parquet(caminho)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_path", F.col("_metadata.file_path"))
    )

    # Aplica o filtro de amostragem em dev/homolog
    if limite_linhas is not None:
        df_raw = df_raw.limit(limite_linhas)

    (
        df_raw.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SCHEMA_RAW}.raw_{entidade}")
    )
    print(f"-> Tabela {SCHEMA_RAW}.raw_{entidade} criada com sucesso.")

# =============================================================================
# 6. INGESTÃO DAS TABELAS DE APOIO E SIMPLES
# =============================================================================
print("Processando tabelas_apoio...")
caminho_apoio = f"{BASE_VOLUME}/tabelas_apoio"

df_apoio = (
    spark.read.parquet(caminho_apoio)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_path", F.col("_metadata.file_path"))
)

# O sufixo _07 (Simples Nacional) é o mais pesado dentro de tabelas_apoio;
# se estiver em dev/homolog, limita-se também a amostragem
if limite_linhas is not None:
    df_apoio = df_apoio.limit(limite_linhas)

(
    df_apoio.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SCHEMA_RAW}.raw_tabelas_apoio")
)
print(f"-> Tabela {SCHEMA_RAW}.raw_tabelas_apoio criada com sucesso.")
print(f"=== CAMADA RAW CONCLUÍDA NO SCHEMA {SCHEMA_RAW} ===")

=== INICIANDO INGESTÃO RAW PARA O AMBIENTE: [PROD] ===
Target Schema Raw: workspace.prod_raw
Processando entidade: empresas a partir de /Volumes/workspace/default/cnpj_ingestion/bronze/empresas...
-> Tabela workspace.prod_raw.raw_empresas criada com sucesso.
Processando entidade: estabelecimentos a partir de /Volumes/workspace/default/cnpj_ingestion/bronze/estabelecimentos...
-> Tabela workspace.prod_raw.raw_estabelecimentos criada com sucesso.
Processando entidade: socios a partir de /Volumes/workspace/default/cnpj_ingestion/bronze/socios...
-> Tabela workspace.prod_raw.raw_socios criada com sucesso.
Processando tabelas_apoio...
-> Tabela workspace.prod_raw.raw_tabelas_apoio criada com sucesso.
=== CAMADA RAW CONCLUÍDA NO SCHEMA workspace.prod_raw ===
